# Experiment 046: GPT-2 XL LoRA pilot (Colab)

Session checks GPT-2 XL parity, LoRA integration, memory and resume behaviour without MMLU test evaluation. Pilot weights are deleted before the card is selected


## 0. Compute units and Drive


In [1]:
NOTEBOOK_NAME = '046_pilot_colab_gpt2_b15_lora'
import glob, json, os, time
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 260.57
unit_rate = 6.77
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

old_sessions = [path for path in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not path.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None or unit_rate is None:
    print('fill cu_before and unit_rate, then rerun this cell')
else:
    for old_path in old_sessions:
        with open(old_path) as handle:
            old = json.load(handle)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as handle:
            handle.write(json.dumps(old) + '\n')
        os.remove(old_path)
    if not os.path.exists(open_path):
        with open(open_path, 'w') as handle:
            json.dump({'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                       'notebook': NOTEBOOK_NAME, 'colab_cu_before': cu_before,
                       'unit_rate': unit_rate}, handle, indent=2)


session 20260826_170744 | cu_before = 260.57
Mounted at /content/drive


## 1. Runtime


In [2]:
import shutil, subprocess, torch

CARD_TO_RUN = 80
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(gpu.strip())
print(f'total vram: {total_gib:.1f} GiB')
assert 'A100' in gpu, 'need an A100 runtime'
if CARD_TO_RUN == 40:
    assert 39 <= total_gib < 48, 'select the A100 40GB runtime first'
elif CARD_TO_RUN == 80:
    assert total_gib >= 79, 'select the A100 80GB runtime'
else:
    raise ValueError('CARD_TO_RUN must be 40 or 80')
assert shutil.disk_usage(ROOT).free / 1024**3 >= 45, 'need at least 45 GiB free on Drive'


GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-81e9667b-7ff2-f4e4-154e-6e393dd59f6d)
total vram: 79.3 GiB


## 2. Repository and dependencies


In [3]:
import base64
import getpass
import os
from pathlib import Path
import subprocess
import sys

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


working in: /content/INM363-IndividualProject-Dissertation/001_Code


In [4]:
import gc, os, subprocess, sys, torch

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'trl==1.7.1', 'transformers==5.5.4', 'peft==0.19.1',
                'accelerate==1.14.0', 'datasets==4.8.4', 'wandb', 'pyyaml'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'], check=True)
scratch_package = os.path.join(repo_root, '002_GPT2_From_Scratch')
if not os.path.isdir(scratch_package):
    scratch_package = os.path.join(repo_root, '003_My GPT From Scratch')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', scratch_package], check=True)
sys.path.insert(0, scratch_package)

import gc
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from gpt2_scratch.weights import REFERENCE_ALIAS, REFERENCE_REVISION

GPT2TokenizerFast.from_pretrained(
    REFERENCE_ALIAS,
    revision=REFERENCE_REVISION,
)
reference = GPT2LMHeadModel.from_pretrained(
    REFERENCE_ALIAS,
    revision=REFERENCE_REVISION,
    dtype=torch.float32,
)
del reference
gc.collect()

print('dependencies installed')


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

dependencies installed


In [5]:
import getpass, os

wandb_key = None
try:
    from google.colab import userdata
    wandb_key = userdata.get('WANDB_API_KEY')
except Exception:
    wandb_key = None
if not wandb_key:
    wandb_key = getpass.getpass('W&B API key: ').strip()
assert wandb_key, 'WANDB_API_KEY is required'
os.environ['WANDB_API_KEY'] = wandb_key
wandb_key = None
print('W&B key loaded')


W&B key loaded


## 3. Wiring checks


In [6]:
checks = subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    os.path.join(scratch_package, 'tests'),
    'tests/test_scratch_gpt_pipeline.py',
    'tests/test_042_m124.py',
    'tests/test_046_gpt2_xl.py',
    'tests/test_046_notebooks.py',
    '-k', 'not test_046_notebooks_are_clean_and_unexecuted and not test_046_units_are_blank',
], capture_output=True, text=True)
print(checks.stdout)
if checks.stderr:
    print(checks.stderr)
checks.check_returncode()
print('wiring checks passed')


........................................................................ [ 81%]
................                                                         [100%]
88 passed, 2 deselected in 31.62s

wiring checks passed


## 4. Registered paths


In [7]:
import json, os, shutil, subprocess, sys, time
sys.path.insert(0, os.getcwd())

from config import load_config
from file_utils import read_json
from scripts.grpo_worker import stream_subprocess
from scripts.s12_eval import atomic_write_json, sha256
from scripts.s46_gpt2_xl import (ALLOCATOR_CONFIG, BASE_TAG, PILOT_ROOT,
                                 archive_memory_margin_pilot,
                                 check_registered_config, commit_card_lock,
                                 commit_token_audit, pilot_passes)

assert os.environ.get('PYTORCH_CUDA_ALLOC_CONF') == ALLOCATOR_CONFIG
assert cu_before is not None and unit_rate is not None, 'fill the compute fields first'
cfg = check_registered_config(load_config('configs/046_colab_gpt2_b15_seed0.yaml'))
TECH = os.path.join(RESULTS, PILOT_ROOT)
CARD_DIR = os.path.join(TECH, f'pilot_{CARD_TO_RUN}gb')
BASE_RESULT = os.path.join(RESULTS, BASE_TAG)
BASE_DIR = os.path.join(ROOT, 'checkpoints', BASE_TAG)
BASE_RECORD = os.path.join(BASE_RESULT, '046_base_manifest.json')
PARITY_RECORD = os.path.join(TECH, '046_xl_parity.json')
TOKEN_AUDIT = os.path.join(TECH, '046_token_audit.json')
SOURCE_AUDIT = os.path.join(RESULTS, '042_technical', '042_token_audit.json')
SOURCE_BASE = os.path.join(RESULTS, '042_COLAB_FULL_M124_base', '042_base_manifest.json')
PILOT_SFT = f'/content/046_pilot_sft_{CARD_TO_RUN}gb'
PILOT_MERGED = f'/content/sft_merged'
PILOT_GRPO = f'/content/046_pilot_grpo_{CARD_TO_RUN}gb'
os.makedirs(CARD_DIR, exist_ok=True)

if CARD_TO_RUN == 80:
    FAILED_40_RECORD = os.path.join(TECH, 'pilot_40gb', 'pilot_failure.json')
    failed_40 = read_json(FAILED_40_RECORD)
    assert failed_40.get('failure_kind') == 'memory', '80GB requires the recorded 40GB memory failure'


In [8]:
def run_worker(command, status_path):
    if os.path.exists(status_path):
        old = read_json(status_path)
        if old.get('completed') is True or old.get('error_type') != 'KeyboardInterrupt':
            return old
        os.remove(status_path)
    process = stream_subprocess(command)
    status = read_json(status_path) if os.path.exists(status_path) else {
        'completed': False, 'error_type': 'MissingStatus'}
    status['returncode'] = process.returncode
    return status

def worker(task, status_name, **kwargs):
    command = [sys.executable, '-u', 'scripts/s46_worker.py',
               '--task', task, '--config', 'configs/046_colab_gpt2_b15_seed0.yaml',
               '--status', os.path.join(CARD_DIR, status_name),
               '--result-dir', CARD_DIR]
    for key, value in kwargs.items():
        command.extend(['--' + key.replace('_', '-'), value])
    return run_worker(command, os.path.join(CARD_DIR, status_name))

def require(status, label):
    if status.get('completed') is not True:
        if status.get('cuda_oom'):
            atomic_write_json(os.path.join(CARD_DIR, 'pilot_failure.json'), {
                'completed': True, 'failure_kind': 'memory', 'stage': label,
                'status': status})
        raise RuntimeError(f'{label} failed: {status}')
    return status


## 5. Convert and check GPT-2 XL


In [9]:
if not os.path.exists(BASE_RECORD):
    convert = worker('convert', 'convert_status.json',
                     scratch_dir='/content/046_xl_base_conversion',
                     out_dir=BASE_DIR, record=BASE_RECORD)
    require(convert, 'base conversion')
    shutil.rmtree('/content/046_xl_base_conversion', ignore_errors=True)
base_record = read_json(BASE_RECORD)
print('base committed:', base_record['artifact'])


base committed: /content/drive/MyDrive/inm363/checkpoints/046_COLAB_FULL_GPT2_B15_base


In [10]:
if not os.path.exists(PARITY_RECORD):
    parity = worker('parity', 'parity_status.json',
                    base_record=BASE_RECORD, record=PARITY_RECORD)
    require(parity, 'numerical parity')
print(read_json(PARITY_RECORD))


{'completed': True, 'experiment': '046', 'source': 'openai-community/gpt2-xl', 'source_revision': '15ea56dee5df4983c59b2538573817e1667135e2', 'base_record_sha256': 'dfd8a6bf125f5866f132e06c9a71792d950dad7f9071b2ab28914296e1a57d94', 'max_abs_unpadded_logit_error': 2.288818359375e-05, 'max_abs_padded_logit_error': 2.288818359375e-05, 'abs_loss_error': 0.0, 'max_abs_cached_logit_error': 9.5367431640625e-06, 'reference_cache_error': 2.288818359375e-05, 'scratch_cache_error': 3.0517578125e-05, 'generation_tokens_equal': True, 'generation_diagnostic': {'first_divergence': None}, 'tied_parameter_and_storage': True, 'reload_logit_error': 0.0, 'max_new_tokens': 32}


In [11]:
audit = commit_token_audit(
    cfg, BASE_RECORD, SOURCE_AUDIT, SOURCE_BASE, TOKEN_AUDIT)
assert audit['completion_cap'] == 512
print('token audit committed')


README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

... (72 outputs left out)

Generating test split:   0%|          | 0/126 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/14 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

token audit committed


## 6. LoRA pilot


In [12]:
retry_record = os.path.join(CARD_DIR, 'logit_projection_retry.json')
retry_archive = os.path.join(
    TECH, 'pilot_80gb_before_logit_projection_fix', 'archive_manifest.json')
if CARD_TO_RUN == 80:
    if os.path.exists(retry_record):
        retry = read_json(retry_record)
        if (retry.get('completed') is not True
                or retry.get('archive_manifest_sha256') != sha256(retry_archive)
                or retry.get('model_sha256') != sha256(
                    os.path.join(scratch_package,
                                 'gpt2_scratch', 'model.py'))):
            raise RuntimeError('the 046 logit projection retry record changed')
    elif archive_memory_margin_pilot(CARD_DIR):
        atomic_write_json(retry_record, {
            'completed': True,
            'archive_manifest_sha256': sha256(retry_archive),
            'model_sha256': sha256(os.path.join(
                scratch_package, 'gpt2_scratch', 'model.py')),
        })
        for path in (PILOT_SFT, PILOT_MERGED, PILOT_GRPO):
            shutil.rmtree(path, ignore_errors=True)
        print('memory pilot archived, clean retry ready')

resume_retry_record = os.path.join(CARD_DIR, 'resume_memory_retry.json')
resume_retry_archive = os.path.join(
    TECH, 'pilot_80gb_before_resume_memory_check', 'archive_manifest.json')
if CARD_TO_RUN == 80:
    if os.path.exists(resume_retry_record):
        retry = read_json(resume_retry_record)
        if (retry.get('completed') is not True
                or retry.get('archive_manifest_sha256') != sha256(resume_retry_archive)
                or retry.get('worker_sha256') != sha256(
                    os.path.join(code_root, 'scripts', 's46_worker.py'))
                or retry.get('allocator_config') != ALLOCATOR_CONFIG):
            raise RuntimeError('the 046 resume memory retry record changed')
    elif archive_memory_margin_pilot(
            CARD_DIR, 'pilot_80gb_before_resume_memory_check',
            'resume memory diagnostic'):
        atomic_write_json(resume_retry_record, {
            'completed': True,
            'archive_manifest_sha256': sha256(resume_retry_archive),
            'worker_sha256': sha256(os.path.join(
                code_root, 'scripts', 's46_worker.py')),
            'allocator_config': ALLOCATOR_CONFIG,
        })
        for path in (PILOT_SFT, PILOT_MERGED, PILOT_GRPO):
            shutil.rmtree(path, ignore_errors=True)
        print('resume memory pilot archived, clean diagnostic ready')

pilot_sft_status = os.path.join(CARD_DIR, 'pilot_sft_status.json')
pilot_sft_archive = os.path.join(
    CARD_DIR, 'pilot_sft_status_before_merge_precision_fix.json')
if os.path.exists(pilot_sft_status):
    failed = read_json(pilot_sft_status)
    registered_failure = (
        failed.get('completed') is False
        and failed.get('error_type') == 'IntegrityError'
        and failed.get('error_message') == (
            'experiment 046 adapter and merged logits differ')
        and failed.get('end_step') == 5
        and failed.get('cuda_oom') is False)
    if registered_failure:
        if os.path.exists(pilot_sft_archive):
            assert read_json(pilot_sft_archive) == failed
        else:
            atomic_write_json(pilot_sft_archive, failed)
        os.remove(pilot_sft_status)
        for path in (PILOT_SFT, PILOT_MERGED, PILOT_GRPO,
                     '/content/merge_diagnostic'):
            shutil.rmtree(path, ignore_errors=True)
        print('failed pilot archived, clean retry ready')


resume memory pilot archived, clean diagnostic ready


In [13]:
sft = worker('pilot_sft', 'pilot_sft_status.json',
             base_record=BASE_RECORD, out_dir=PILOT_SFT,
             start_dir=PILOT_MERGED, grpo_dir=PILOT_GRPO)
require(sft, 'pilot SFT')
assert sft['result']['updates'] == 5
assert sft['result']['merge']['verified'] is True


Loading weights: 100% 772/772 [00:00<00:00, 3082.38it/s]
Adding EOS to train dataset: 100% 32/32 [00:00<00:00, 6722.99 examples/s]
Tokenizing train dataset: 100% 32/32 [00:00<00:00, 812.06 examples/s]
Building labels for train dataset: 100% 32/32 [00:00<00:00, 3052.48 examples/s]
{'time': '17:12:33', 'loss': '2.388', 'grad_norm': '0.9904', 'learning_rate': '0.0001', 'entropy': '1.972', 'mean_token_accuracy': '0.5607', 'num_tokens': '3391', 'epoch': '0.5'}
{'time': '17:12:34', 'loss': '2.384', 'grad_norm': '1.086', 'learning_rate': '9.98e-05', 'entropy': '1.898', 'mean_token_accuracy': '0.553', 'num_tokens': '6776', 'epoch': '1'}
{'time': '17:12:35', 'loss': '2.258', 'grad_norm': '1.103', 'learning_rate': '9.96e-05', 'entropy': '1.963', 'mean_token_accuracy': '0.5535', 'num_tokens': '1.03e+04', 'epoch': '1.5'}
{'time': '17:12:37', 'loss': '2.162', 'grad_norm': '1.23', 'learning_rate': '9.94e-05', 'entropy': '1.87', 'mean_token_accuracy': '0.5784', 'num_tokens': '1.355e+04', 'epoch': '2'

In [ ]:
phase_a = worker('pilot_grpo_a', 'pilot_grpo_a_status.json',
                     start_dir=PILOT_MERGED, out_dir=PILOT_GRPO,
                     precision_report=os.path.join(CARD_DIR, 'precision_a.json'))
require(phase_a, 'pilot GRPO phase A')
assert phase_a['end_step'] >= 15

phase_b = worker('pilot_grpo_b', 'pilot_grpo_b_status.json',
                 start_dir=PILOT_MERGED, out_dir=PILOT_GRPO,
                 precision_report=os.path.join(CARD_DIR, 'precision_b.json'))
require(phase_b, 'pilot GRPO phase B')
assert phase_b['end_step'] >= 30


Streaming output truncated to the last 5000 lines.
│ │               │ Fans, Culture │               │              │           │ │
│ │               │ & Society 21, │               │              │           │ │
│ │               │ 74-97         │               │              │           │ │
│ │               │ (author:      │               │              │           │ │
│ │               │ Kalisch,      │               │              │           │ │
│ │               │ R.D., Berlin, │               │              │           │ │
│ │               │ A.E., FORM +  │               │              │           │ │
│ │               │ STIMPUCCI     │               │              │           │ │
│ │               │ 83DO:260 and  │               │              │           │ │
│ │               │ 86DO:193)     │               │              │           │ │
│ │               │               │               │              │           │ │
│ │               │               │               │       

## 7. Close the pilot


In [ ]:
statuses = (sft, phase_a, phase_b)
report = {
    'completed': True,
    'experiment': '046',
    'card_requested': CARD_TO_RUN,
    'sft_updates': sft['result']['updates'],
    'grpo_updates': phase_b['result']['updates'],
    'peak_reserved_fraction': max(row.get('peak_reserved_fraction', 0) for row in statuses),
    'peak_reserved_gib': max(row.get('peak_reserved_gib', 0) for row in statuses),
    'peak_allocated_fraction': max(row.get('peak_allocated_fraction', 0) for row in statuses),
    'peak_allocated_gib': max(row.get('peak_allocated_gib', 0) for row in statuses),
    'allocator_config': phase_b['allocator_config'],
    'resume_memory': phase_b['result']['precision']['resume_memory'],
    'total_vram_gib': phase_b['total_vram_gib'],
    'adapter_reload_verified': sft['result']['adapter_reload_verified'],
    'merge_verified': sft['result']['merge']['verified'],
    'resume_verified': phase_b['result']['resume_verified'],
    'sft_lora_setup': sft['result']['setup'],
    'precision_audit': phase_b['result']['precision'],
    'lora_setup': phase_b['result']['precision']['lora_setup'],
    'scientific_metrics_used': False,
    'mmlu_test_passes': 0,
}
assert all(row.get('allocator_config') == ALLOCATOR_CONFIG for row in statuses)
if CARD_TO_RUN == 80:
    report['failed_40_sha256'] = sha256(FAILED_40_RECORD)
report_path = os.path.join(CARD_DIR, 'pilot_report.json')
atomic_write_json(report_path, report)
for path in (PILOT_SFT, PILOT_MERGED, PILOT_GRPO):
    shutil.rmtree(path, ignore_errors=True)
closeout_path = os.path.join(CARD_DIR, 'pilot_closeout.json')
atomic_write_json(closeout_path, {
    'completed': True,
    'pilot_report_sha256': sha256(report_path),
    'weights_deleted': not any(os.path.exists(path) for path in (
        PILOT_SFT, PILOT_MERGED, PILOT_GRPO)),
})
closeout = read_json(closeout_path)
if not pilot_passes(report, closeout):
    if report['peak_reserved_fraction'] > 0.95:
        atomic_write_json(os.path.join(CARD_DIR, 'pilot_failure.json'), {
            'completed': True, 'failure_kind': 'memory', 'stage': 'memory margin',
            'report_sha256': sha256(report_path)})
    raise RuntimeError('the pilot did not pass')
lock = commit_card_lock(ROOT, report_path, closeout_path)
print('science card:', lock['card'])


science card: A100_80GB


## 8. Close compute units


In [ ]:
cu_after = 245.58
print('cu_after =', cu_after if cu_after is not None else 'FILL ME')
if cu_after is None:
    print('fill cu_after, then rerun this cell')
elif os.path.exists(open_path):
    with open(open_path) as handle:
        record = json.load(handle)
    record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
    record['colab_cu_after'] = cu_after
    record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
    record['completed'] = True
    with open(UNITS_LOG, 'a') as handle:
        handle.write(json.dumps(record) + '\n')
    os.remove(open_path)
    print('logged:', record)


cu_after = 270.33
logged: {'session_id': '20260826_170744', 'ts_start': '2026-08-26 17:08', 'notebook': '046_pilot_colab_gpt2_b15_lora', 'colab_cu_before': 260.57, 'unit_rate': 6.77, 'ts_end': '2026-08-26 17:38', 'colab_cu_after': 270.33, 'colab_cu_used': -9.76, 'completed': True}
